# Week 7 — Advanced Models

Loads the leakage-safe, feature-engineered `month=6` train/test split exported at the end of
`05_data_cleaning_and_retraining.ipynb` (Week 6) and:

- Trains a gradient boosting model (XGBoost or LightGBM if available, else sklearn's
  `HistGradientBoostingRegressor` as a fallback).
- Runs light hyperparameter tuning over `max_depth`, `learning_rate`, and `n_estimators`.
- Reports test-set metrics for the tuned model, and compares against the Week 6 baselines.

No feature engineering happens in this notebook — that's already been done upstream in Week 6,
using train-only fitting to avoid leakage. This notebook only consumes the already-processed,
already-split data.


## 1. Imports

In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Prefer XGBoost, then LightGBM, then fall back to sklearn's own gradient boosting
# implementation so this notebook still runs even if neither external library is installed.
try:
    from xgboost import XGBRegressor
    GB_LIBRARY = "xgboost"
except ImportError:
    try:
        from lightgbm import LGBMRegressor
        GB_LIBRARY = "lightgbm"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        GB_LIBRARY = "sklearn"

print(f"Using gradient boosting library: {GB_LIBRARY}")


## 2. Load the exported Week 6 data

Expects `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`, and `column_lists.json` in
`data/processed/` (written by the export cell added at the end of Week 6's notebook, right
after the `month=6` run).

In [ ]:
DATA_DIR = "data/processed"

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")
y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze("columns")
y_test = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze("columns")

with open(f"{DATA_DIR}/column_lists.json") as f:
    col_lists = json.load(f)
numeric_cols = col_lists["numeric_cols"]
categorical_cols = col_lists["categorical_cols"]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"{len(numeric_cols)} numeric cols, {len(categorical_cols)} categorical cols")


## 3. Metrics helper (same definitions as Week 6, for apples-to-apples comparison)

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
        "mdape": np.median(np.abs((y_true - y_pred) / y_true)),
    }


## 4. Build the gradient boosting pipeline

Numeric columns pass through as-is (gradient boosting models don't need scaling); low-cardinality
categorical columns are one-hot encoded, same as Week 6. `build_regressor()` maps the same
`n_estimators` / `max_depth` / `learning_rate` names onto whichever library is available, so the
rest of the notebook doesn't need to branch on `GB_LIBRARY` again.

In [ ]:
def build_regressor(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42):
    if GB_LIBRARY == "xgboost":
        return XGBRegressor(
            n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
            random_state=random_state, n_jobs=-1,
        )
    elif GB_LIBRARY == "lightgbm":
        return LGBMRegressor(
            n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
            random_state=random_state, n_jobs=-1, verbose=-1,
        )
    else:  # sklearn fallback -- HistGradientBoostingRegressor uses max_iter, not n_estimators,
           # and -1 for max_depth means "unlimited" instead of None
        return HistGradientBoostingRegressor(
            max_iter=n_estimators, max_depth=(max_depth if max_depth else None),
            learning_rate=learning_rate, random_state=random_state,
        )


def build_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )
    return Pipeline(steps=[("preprocessing", preprocessor), ("model", model)])


## 5. Baseline gradient boosting run (default hyperparameters)

Sanity check before tuning -- confirms the pipeline runs end to end and gives a reference point
for whether tuning actually helps.

In [ ]:
baseline_pipeline = build_pipeline(build_regressor())
baseline_pipeline.fit(X_train, y_train)
baseline_pred = baseline_pipeline.predict(X_test)

baseline_metrics = compute_metrics(y_test, baseline_pred)
baseline_metrics["model"] = f"{GB_LIBRARY} (default params)"
print(baseline_metrics)


## 6. Light hyperparameter tuning

Small randomized search over `n_estimators`, `max_depth`, and `learning_rate` -- the three knobs
called out in the Week 7 brief. Tuned against `X_train`/`y_train` only, via internal cross-validation
(`cv=3`), so the held-out `X_test`/`y_test` stays untouched until final evaluation.

Note: the sklearn fallback (`HistGradientBoostingRegressor`) names its "number of trees" parameter
`max_iter` rather than `n_estimators`, so the grid's key is chosen based on `GB_LIBRARY`.

In [ ]:
n_estimators_param = "model__n_estimators" if GB_LIBRARY in ("xgboost", "lightgbm") else "model__max_iter"

param_distributions = {
    n_estimators_param: [100, 200, 300, 400, 600],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1],
}

search_pipeline = build_pipeline(build_regressor())

search = RandomizedSearchCV(
    search_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV R²:", search.best_score_)
print("Best params:", search.best_params_)


## 7. Evaluate the tuned model on the held-out test set

In [ ]:
best_pipeline = search.best_estimator_
tuned_pred = best_pipeline.predict(X_test)

tuned_metrics = compute_metrics(y_test, tuned_pred)
tuned_metrics["model"] = f"{GB_LIBRARY} (tuned)"

results_df = pd.DataFrame([baseline_metrics, tuned_metrics])[["model", "r2", "mae", "mape", "mdape"]]
results_df


## 8. Feature importance (tuned model)

Same sanity check as Week 6 -- confirms leakage columns still don't show up, and shows what the
gradient boosting model is keying off of vs. the tree models from Week 6.

In [ ]:
def get_feature_names(pipeline):
    return pipeline.named_steps["preprocessing"].get_feature_names_out()

feature_names = get_feature_names(best_pipeline)
model_step = best_pipeline.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    importances = model_step.feature_importances_
    top_features = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(15)
    )
    print(top_features.to_string(index=False))
else:
    print(f"{GB_LIBRARY} model has no feature_importances_ attribute to inspect.")


## 9. Save the tuned model + predictions

Persists the tuned pipeline so Week 8's evaluation notebook can load it directly rather than
retraining.

In [ ]:
import os
import joblib

os.makedirs("data/processed", exist_ok=True)
joblib.dump(best_pipeline, "data/processed/best_gb_model.joblib")

test_predictions_df = pd.DataFrame({"y_true": y_test.values, "y_pred": tuned_pred})
test_predictions_df.to_csv("data/processed/test_predictions.csv", index=False)

print("Saved data/processed/best_gb_model.joblib and data/processed/test_predictions.csv")
